# STARRT.ai experiments

Runs TF-IDF, Sentence-BERT + LR, DistilBERT and a zero-shot LLM on the same fixed, leakage-free folds.

**Before running:** Settings → Accelerator = **GPU T4 x2** (or P100), Internet = **On**. Add the uploaded `starrt-kaggle` dataset with *Add Input*.

Results are saved in `/kaggle/working/results/` and zipped at the end as `starrt_results.zip`.

In [ ]:
import glob, os, subprocess, sys
hits = glob.glob('/kaggle/input/**/run_sbert.py', recursive=True)
assert hits, 'Dataset not attached: use Add Input and pick the starrt-kaggle dataset'
CODE = os.path.dirname(hits[0])
print('code folder:', CODE)
import torch; print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

def run(script, args=''):
    cmd = f'cd /kaggle/working && python {CODE}/{script} {args}'
    print('>>', cmd); r = subprocess.run(cmd, shell=True); assert r.returncode == 0, f'{script} failed'

In [ ]:
# packages (Kaggle usually has them already)
!pip -q install -U sentence-transformers 2>/dev/null | tail -1

## 1. Quick smoke test (2–3 min)
Checks that everything works before the long runs. If this cell fails, stop and send the error.

In [ ]:
run('run_distilbert.py', '--datasets faruque --folds 0 --epochs 1 --limit 400 --skip_forms --tag smoke_distilbert')
run('run_llm_zeroshot.py', '--datasets faruque --limit 3 --tag smoke_llm')
print('smoke test OK')

## 2. TF-IDF baselines (CPU, ~15 min)

In [ ]:
run('run_tfidf.py')

## 3. Sentence-BERT + Logistic Regression (~15 min)
`all-MiniLM-L6-v2` is the model the original STARRT.ai used; `all-mpnet-base-v2` is stronger.

In [ ]:
run('run_sbert.py')

## 4. DistilBERT fine-tuning (~1.5 h)
If time is short, run Djinni only first: `--datasets djinni`.

In [ ]:
run('run_distilbert.py')

## 5. Zero-shot LLM, Qwen2.5-3B-Instruct (~30 min)

In [ ]:
run('run_llm_zeroshot.py')

## 6. Zip results
Download `starrt_results.zip` from the Output panel on the right and upload it in the chat.

In [ ]:
import shutil, glob
for f in glob.glob('/kaggle/working/results/emb_*.npy'): os.remove(f)   # embeddings are large and not needed
for f in glob.glob('/kaggle/working/results/smoke_*'): os.remove(f)
shutil.make_archive('/kaggle/working/starrt_results', 'zip', '/kaggle/working/results')
print(sorted(os.listdir('/kaggle/working/results')))